# Predict Customer Churn
## Score: .91427

In [1]:
N_FOLDS = 10
N_SEEDS = 5
OPTUNA_TRIALS = 60
USE_RF_ET = False
USE_ORIGINAL_DATA = True
TARGET_ENC_M = 20

In [2]:
import numpy as np
import pandas as pd

train = pd.read_csv("playground-series-s6e3/train.csv")
test = pd.read_csv("playground-series-s6e3/test.csv")
test_ids = test['id']
X_test_raw = test.drop(columns=['id'])

if USE_ORIGINAL_DATA:
    original = pd.read_csv("original_data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
    original = original.drop(columns=['customerID'])
    original = original[train.columns.drop('id')]
    train_combined = pd.concat([train.drop(columns=['id']), original], ignore_index=True)
    sample_weights = np.array([1.0] * len(train) + [0.5] * len(original))
else:
    train_combined = train.drop(columns=['id'])
    sample_weights = np.ones(len(train_combined))

y_full = train_combined['Churn'].map({'Yes': 1, 'No': 0})
X_full = train_combined.drop(columns=['Churn'])
print(f"Train: {len(X_full)}, Original: {USE_ORIGINAL_DATA}")

Train: 601237, Original: True


In [3]:
def engineer_features(df, total_charges_median=None):
    df = df.copy()
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    median_tc = total_charges_median if total_charges_median is not None else df['TotalCharges'].median()
    df['TotalCharges'] = df['TotalCharges'].fillna(median_tc)
    df['AvgChargesPerMonth'] = df['TotalCharges'] / df['tenure'].replace(0, 1)
    df['MonthlyChargesSqrt'] = np.sqrt(df['MonthlyCharges'])
    df['tenure_squared'] = df['tenure'] ** 2
    df['ChargesRatio'] = df['TotalCharges'] / (df['MonthlyCharges'] * (df['tenure'] + 1))
    df['Contract_MonthToMonth'] = (df['Contract'] == 'Month-to-month').astype(int)
    df['Contract_OneYear'] = (df['Contract'] == 'One year').astype(int)
    df['Contract_TwoYear'] = (df['Contract'] == 'Two year').astype(int)
    df['IsFiberOptic'] = (df['InternetService'] == 'Fiber optic').astype(int)
    df['NumStreaming'] = ((df['StreamingTV'] == 'Yes') | (df['StreamingMovies'] == 'Yes')).astype(int)
    df['NumStreamingBoth'] = ((df['StreamingTV'] == 'Yes') & (df['StreamingMovies'] == 'Yes')).astype(int)
    addon_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
    df['NumAddons'] = sum((df[c] == 'Yes').astype(int) for c in addon_cols)
    df['PaymentElectronic'] = (df['PaymentMethod'] == 'Electronic check').astype(int)
    df['HasDependents'] = (df['Dependents'] == 'Yes').astype(int)
    df['HasPartner'] = (df['Partner'] == 'Yes').astype(int)
    df['SeniorWithShortTenure'] = (df['SeniorCitizen'] == 1) & (df['tenure'] < 12)
    df['SeniorWithShortTenure'] = df['SeniorWithShortTenure'].astype(int)
    df['HighChargeShortTenure'] = (df['MonthlyCharges'] > 70) & (df['tenure'] < 12)
    df['HighChargeShortTenure'] = df['HighChargeShortTenure'].astype(int)
    return df

def target_encode(train_df, test_df, col, target_series, m=5):
    global_mean = target_series.mean()
    combined = pd.DataFrame({col: train_df[col], '_y': target_series.values})
    agg = combined.groupby(col)['_y'].agg(['mean', 'count'])
    smooth = (agg['mean'] * agg['count'] + global_mean * m) / (agg['count'] + m)
    train_df = train_df.copy()
    test_df = test_df.copy()
    train_df[f'{col}_te'] = train_df[col].map(smooth).fillna(global_mean)
    test_df[f'{col}_te'] = test_df[col].map(smooth).fillna(global_mean)
    return train_df, test_df

X_full = engineer_features(X_full)
tc_median = X_full['TotalCharges'].median()
X_test_raw = engineer_features(X_test_raw, total_charges_median=tc_median)

for col in ['Contract', 'PaymentMethod', 'InternetService']:
    X_full, X_test_raw = target_encode(X_full, X_test_raw, col, y_full, m=TARGET_ENC_M)
    X_full = X_full.drop(columns=[col])
    X_test_raw = X_test_raw.drop(columns=[col])

X_encoded = pd.get_dummies(X_full, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_raw, drop_first=True)
X_test_encoded = X_test_encoded.reindex(columns=X_encoded.columns, fill_value=0)

In [4]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fit_kw = {'sample_weight': sample_weights}

In [5]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.95),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 4.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    scores = []
    for train_idx, val_idx in cv.split(X_encoded, y_full):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
        sw_tr = sample_weights[train_idx]
        m = XGBClassifier(**params, random_state=42, eval_metric='logloss')
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        scores.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
    return np.mean(scores)

study_xgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(n_startup_trials=10))
study_xgb.optimize(xgb_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
best_xgb_params = study_xgb.best_params
print(f"XGB Best CV AUC: {study_xgb.best_value:.4f}")

[I 2026-03-08 23:04:17,534] A new study created in memory with name: no-name-f46976f3-1b06-43fd-9c3d-bb7d76282046


  0%|          | 0/60 [00:00<?, ?it/s]

[I 2026-03-08 23:05:58,120] Trial 0 finished with value: 0.9138445454628451 and parameters: {'n_estimators': 445, 'max_depth': 4, 'learning_rate': 0.022181869094077666, 'subsample': 0.7436796953080804, 'colsample_bytree': 0.8602434879682844, 'scale_pos_weight': 2.987041136729267, 'reg_alpha': 6.684640618393725, 'reg_lambda': 0.004111662004087989, 'min_child_weight': 10}. Best is trial 0 with value: 0.9138445454628451.
[I 2026-03-08 23:09:21,842] Trial 1 finished with value: 0.9157205922770913 and parameters: {'n_estimators': 784, 'max_depth': 6, 'learning_rate': 0.05557219890290159, 'subsample': 0.7552279795585288, 'colsample_bytree': 0.6249329187229254, 'scale_pos_weight': 1.7710259922260532, 'reg_alpha': 0.28422171332567686, 'reg_lambda': 1.0436474511678189, 'min_child_weight': 2}. Best is trial 1 with value: 0.9157205922770913.
[I 2026-03-08 23:11:49,773] Trial 2 finished with value: 0.915351109196599 and parameters: {'n_estimators': 624, 'max_depth': 5, 'learning_rate': 0.148192166

In [6]:
from lightgbm import LGBMClassifier

def lgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.95),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 4.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
    }
    scores = []
    for train_idx, val_idx in cv.split(X_encoded, y_full):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
        sw_tr = sample_weights[train_idx]
        m = LGBMClassifier(**params, random_state=42, verbose=-1)
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        scores.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
    return np.mean(scores)

study_lgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(n_startup_trials=10))
study_lgb.optimize(lgb_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
best_lgb_params = study_lgb.best_params
print(f"LGB Best CV AUC: {study_lgb.best_value:.4f}")

[I 2026-03-09 01:25:51,868] A new study created in memory with name: no-name-b9ea7c35-13f7-4771-be5e-8da0ce4c0061


  0%|          | 0/60 [00:00<?, ?it/s]

c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python313\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python3

[I 2026-03-09 01:26:39,150] Trial 0 finished with value: 0.9149084770918654 and parameters: {'n_estimators': 407, 'max_depth': 3, 'learning_rate': 0.08610927544436575, 'subsample': 0.627452402098211, 'colsample_bytree': 0.825271635660718, 'scale_pos_weight': 3.685026561992249, 'reg_alpha': 0.17049930910940939, 'reg_lambda': 0.02206677687654667, 'min_child_samples': 59}. Best is trial 0 with value: 0.9149084770918654.
[I 2026-03-09 01:28:27,637] Trial 1 finished with value: 0.9157455491716975 and parameters: {'n_estimators': 558, 'max_depth': 7, 'learning_rate': 0.09984552247014303, 'subsample': 0.8811207518724, 'colsample_bytree': 0.8706978966803492, 'scale_pos_weight': 2.4235492974780017, 'reg_alpha': 7.11706532940151, 'reg_lambda': 0.27081653970383085, 'min_child_samples': 6}. Best is trial 1 with value: 0.9157455491716975.
[I 2026-03-09 01:29:58,225] Trial 2 finished with value: 0.9157985861752094 and parameters: {'n_estimators': 488, 'max_depth': 8, 'learning_rate': 0.0876044548415

In [7]:
from catboost import CatBoostClassifier

def cat_objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 300, 800),
        'depth': trial.suggest_int('depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.6, 0.95),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 4.0),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
    }
    scores = []
    for train_idx, val_idx in cv.split(X_encoded, y_full):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
        sw_tr = sample_weights[train_idx]
        m = CatBoostClassifier(**params, random_state=42, verbose=0)
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        scores.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
    return np.mean(scores)

study_cat = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(n_startup_trials=10))
study_cat.optimize(cat_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
best_cat_params = study_cat.best_params
print(f"CatBoost Best CV AUC: {study_cat.best_value:.4f}")

[I 2026-03-09 03:20:46,707] A new study created in memory with name: no-name-5f61188a-68a4-4139-9bbf-65ec364cb02e


  0%|          | 0/60 [00:00<?, ?it/s]

[I 2026-03-09 03:25:48,997] Trial 0 finished with value: 0.9155249897185677 and parameters: {'iterations': 723, 'depth': 7, 'learning_rate': 0.10641558727578224, 'subsample': 0.659432786644122, 'colsample_bylevel': 0.8335110371490138, 'scale_pos_weight': 2.1107220896158396, 'l2_leaf_reg': 0.7001254018572419}. Best is trial 0 with value: 0.9155249897185677.
[I 2026-03-09 03:28:26,683] Trial 1 finished with value: 0.9139600739347614 and parameters: {'iterations': 559, 'depth': 3, 'learning_rate': 0.04836291729218233, 'subsample': 0.9160056659105259, 'colsample_bylevel': 0.6782575242550477, 'scale_pos_weight': 1.8679059141560224, 'l2_leaf_reg': 0.0011173606938134416}. Best is trial 0 with value: 0.9155249897185677.
[I 2026-03-09 03:30:53,059] Trial 2 finished with value: 0.9152415507314793 and parameters: {'iterations': 539, 'depth': 3, 'learning_rate': 0.14546354620714164, 'subsample': 0.7102969997859561, 'colsample_bylevel': 0.7889200998353564, 'scale_pos_weight': 3.422201514995664, 'l2

In [8]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize

n_models = 3 if not USE_RF_ET else 5
oof = np.zeros((len(X_encoded), n_models))
test_preds = np.zeros((len(X_test_encoded), n_models))

def get_models(seed, fold):
    models = [
        XGBClassifier(**best_xgb_params, random_state=seed+fold, eval_metric='logloss'),
        LGBMClassifier(**best_lgb_params, random_state=seed+fold, verbose=-1),
        CatBoostClassifier(**best_cat_params, random_state=seed+fold, verbose=0),
    ]
    if USE_RF_ET:
        models.extend([
            RandomForestClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=seed+fold),
            ExtraTreesClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=seed+fold),
        ])
    return models

for fold, (train_idx, val_idx) in enumerate(cv.split(X_encoded, y_full)):
    X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
    y_tr = y_full.iloc[train_idx]
    sw_tr = sample_weights[train_idx]
    models = get_models(42, fold)
    for i, m in enumerate(models):
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        oof[val_idx, i] = m.predict_proba(X_val)[:, 1]
        test_preds[:, i] += m.predict_proba(X_test_encoded)[:, 1] / N_FOLDS

def neg_auc(w):
    blend = oof @ w
    return -roc_auc_score(y_full, blend)

best_auc = -1
best_w = None
for x0 in [np.ones(n_models)/n_models, np.array([0.5, 0.3, 0.2] if n_models==3 else [0.2]*5)]:
    result = minimize(neg_auc, x0=x0, method='SLSQP',
                      bounds=[(0, 1)]*n_models,
                      constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
    auc = -result.fun
    if auc > best_auc:
        best_auc = auc
        best_w = result.x
blend_weights = best_w
blend_oof = oof @ blend_weights
names = ['XGB','LGB','Cat'] + (['RF','ET'] if USE_RF_ET else [])
print(f"Blend OOF AUC: {roc_auc_score(y_full, blend_oof):.4f}")
print(f"Weights: {dict(zip(names, np.round(blend_weights, 4)))}")

Blend OOF AUC: 0.9161
Weights: {'XGB': np.float64(0.3333), 'LGB': np.float64(0.3333), 'Cat': np.float64(0.3333)}


In [9]:
all_test_preds = [test_preds @ blend_weights]
for base_seed in [123, 456, 789, 2024, 2025][:N_SEEDS-1]:
    tp = np.zeros((len(X_test_encoded), n_models))
    for fold, (train_idx, val_idx) in enumerate(cv.split(X_encoded, y_full)):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr = y_full.iloc[train_idx]
        sw_tr = sample_weights[train_idx]
        models = get_models(base_seed, fold)
        for i, m in enumerate(models):
            m.fit(X_tr, y_tr, sample_weight=sw_tr)
            tp[:, i] += m.predict_proba(X_test_encoded)[:, 1] / N_FOLDS
    all_test_preds.append(tp @ blend_weights)

final_preds = np.mean(all_test_preds, axis=0)
submission = pd.DataFrame({'id': test_ids, 'Churn': final_preds})
submission.to_csv('submission.csv', index=False)
submission.head()

,id,Churn
0,594194,0.105364
1,594195,0.001179
2,594196,0.146266
3,594197,0.005516
4,594198,0.588684
